# 00 — Reproducción completa

Este notebook es la puerta de entrada al proyecto. Reconstruye el
análisis desde las respuestas crudas locales, actualiza la
trazabilidad, ejecuta las pruebas y compara las conclusiones antes y
después.

El pipeline respeta el calendario efectivo de publicación: para una
emisión en el mes `t`, solo admite variables con `available_date`
anterior o igual a `forecast_issue_date`.

In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

print(f"Raíz del proyecto: {ROOT}")

Raíz del proyecto: C:\Users\litoa\OneDrive\Documents\GitHub\ml_finance_carenas\Morosidad_bancaria_arenas


## Inventario mínimo

La reproducción local no descarga datos ni necesita credenciales.
Requiere las capas `data/raw`, el código de `src` y las
configuraciones versionadas en `configs`.

In [2]:
required = [
    ROOT / "data" / "raw" / "bcch" / "catalog_monthly.json",
    ROOT / "data" / "raw" / "cmf_publication_calendar" / "cmf_press.html",
    ROOT / "configs" / "base.toml",
    ROOT / "src" / "morosidad_bancaria" / "cli.py",
]
inventory = {path.relative_to(ROOT).as_posix(): path.exists() for path in required}
inventory

{'data/raw/bcch/catalog_monthly.json': True,
 'data/raw/cmf_publication_calendar/cmf_press.html': True,
 'configs/base.toml': True,
 'src/morosidad_bancaria/cli.py': True}

## Ejecución

Para volver a correr las doce etapas desde este notebook, cambie
`RUN_FULL_REPRODUCTION` a `True`. La ejecución conserva cerrado el
holdout 2024–2025.

In [3]:
import os
import subprocess
import sys

RUN_FULL_REPRODUCTION = False

if RUN_FULL_REPRODUCTION:
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(ROOT / "src")
    environment["PYTHONDONTWRITEBYTECODE"] = "1"
    subprocess.run(
        [sys.executable, "-B", "scripts/run_reproduction.py"],
        cwd=ROOT,
        env=environment,
        check=True,
    )
else:
    print("Ejecución omitida. Use RUN_FULL_REPRODUCTION = True para reconstruir.")

Ejecución omitida. Use RUN_FULL_REPRODUCTION = True para reconstruir.


## Resultado de la última reproducción

In [4]:
import json
from IPython.display import Markdown, display

comparison_path = ROOT / "reports" / "reproduction_comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    verdict = "NO CAMBIARON" if not comparison["conclusions_changed"] else "CAMBIARON"
    display(Markdown(
        f"**Conclusiones: {verdict}.**  "
        f"Archivos analíticos modificados: {comparison['key_files_changed']}."
    ))
    comparison["after"]
else:
    print("Aún no existe una comparación. Ejecute scripts/run_reproduction.py.")

**Conclusiones: NO CAMBIARON.**  Archivos analíticos modificados: 0.